# TD6 · Taller · Detección y tracking con YOLO26

**Dataset `items-escritorio` · Versión para completar**

Vamos a hacer en código lo mismo que hicimos en clase en Ultralytics Platform: entrenar **YOLO26n** sobre el dataset que etiquetamos y después usarlo para **seguir objetos en un video** que simula la cinta de una caja.

Recorrido: **dataset → entrenamiento → resultados → tracking en video**.

**Solo hay que completar las dos líneas de la sección 4.** El resto ya está resuelto: úsenlo como punto de partida para el TP.

En Colab activen una GPU en **Entorno de ejecución → Cambiar tipo de entorno de ejecución → T4 GPU**. Ejecuten las celdas en orden.

## 1. Dataset

**a)** Instalamos Ultralytics. `lap` es una dependencia que usa el tracker.

In [1]:
%pip install -q ultralytics lap


[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


**b)** El dataset ya está etiquetado y es público en Ultralytics Platform: [platform.ultralytics.com/lucian5102/datasets/items-escritorio](https://platform.ultralytics.com/lucian5102/datasets/items-escritorio).

Para descargarlo desde Python, Ultralytics pide una API key. Créenla en su cuenta, en **Settings → API Keys**, y péguenla en el campo oculto que aparece al ejecutar la celda. No la escriban en el código.

La celda baja las imágenes y las etiquetas a una carpeta local en formato YOLO y devuelve la ruta a su `data.yaml`. La descarga es asíncrona: por eso se usa `await`.

En el TP van a cambiar la URI por la de **su** dataset, `ul://<usuario>/datasets/<nombre>`.

In [ ]:
import os
from getpass import getpass
from pathlib import Path

from IPython.display import Image, Video, display
from ultralytics import YOLO
from ultralytics.data.converter import convert_ndjson_to_yolo

os.environ["ULTRALYTICS_API_KEY"] = getpass("API key de Ultralytics: ")
DATASET = await convert_ndjson_to_yolo("ul://lucian5102/datasets/items-escritorio")
print(f"Dataset listo en: {DATASET}")

## 2. Entrenamiento

Partimos de **YOLO26n** preentrenado en COCO y lo ajustamos a nuestras clases (*fine-tuning*). Son los mismos parámetros que usamos en la Platform: 100 épocas, imágenes de 640 px y `batch=-1`, que elige solo el tamaño de lote según la memoria de la GPU.

Con 33 imágenes tarda unos minutos en una T4. En CPU funciona, pero bastante más lento.

In [ ]:
model = YOLO("yolo26n.pt")
model.train(data=DATASET, epochs=100, imgsz=640, batch=-1, name="items-escritorio")

carpeta = Path(model.trainer.save_dir)
BEST = carpeta / "weights" / "best.pt"
print(f"Pesos del mejor modelo: {BEST}")

## 3. Resultados

**a)** Evaluamos el mejor modelo sobre validación. **mAP50** cuenta un acierto si la caja predicha se superpone al menos 50 % con la real; **mAP50-95** promedia umbrales más exigentes, del 50 % al 95 %.

In [ ]:
metricas = model.val()

print(f"mAP50:     {metricas.box.map50:.3f}")
print(f"mAP50-95:  {metricas.box.map:.3f}")
print(f"Precision: {metricas.box.mp:.3f}")
print(f"Recall:    {metricas.box.mr:.3f}")

print("\nmAP50-95 por clase:")
for i, nombre in model.names.items():
    print(f"  {nombre:<20} {metricas.box.maps[i]:.3f}")

**b)** Ultralytics guarda los gráficos del entrenamiento en la carpeta del run: curvas de pérdida y métricas por época, y la matriz de confusión.

In [ ]:
display(Image(carpeta / "results.png", width=900))
display(Image(carpeta / "confusion_matrix.png", width=600))

**c)** Predicciones del modelo sobre un lote de validación.

In [ ]:
display(Image(carpeta / "val_batch0_pred.jpg", width=700))

**Para pensar**

1. ¿Por qué con tan pocas imágenes las métricas de validación pueden salir casi perfectas? ¿Qué nos dicen y qué no?
2. Miren la matriz de confusión: ¿qué clases se confunden entre sí?

**Sus respuestas:**

1. …
2. …

## 4. Tracking en video

El video `cinta.mp4` simula una cinta transportadora: los objetos entran y salen de la escena. Si no lo subieron al panel de archivos de Colab, la celda de abajo muestra un botón para subirlo.

Detectar no alcanza: en cada frame el modelo encuentra cajas, pero no sabe que la caja del frame 10 es el mismo objeto que la del frame 11. El **tracker** une las detecciones entre frames y le asigna a cada objeto un **ID** que se mantiene mientras está en escena. Así una caja registradora cobraría cada producto una sola vez.

**a)** Completen las dos líneas:

1. Carguen el modelo entrenado desde `BEST`.
2. Corran el tracking sobre `VIDEO` con **BoT-SORT**, el tracker que pide el TP.

*Pista:* `YOLO(ruta)` carga los pesos. `modelo.track(source=..., tracker=..., stream=True)` devuelve un resultado por frame. El archivo de BoT-SORT es `"botsort.yaml"`. Hay que pasarlo explícitamente: en esta versión de Ultralytics el tracker por defecto es otro, TrackTrack. ByteTrack es `"bytetrack.yaml"`.

In [ ]:
VIDEO = "cinta.mp4"
if not Path(VIDEO).exists():  # en Colab aparece un botón para subir el video
    from google.colab import files
    files.upload()

modelo_tracking = ...  # COMPLETAR: cargar los pesos de BEST
resultados = ...  # COMPLETAR: tracking sobre VIDEO con BoT-SORT y stream=True

**b)** Esta función ya está resuelta. Recorre los resultados y arma un video con la caja de cada objeto, su **ID**, su clase, la **probabilidad** de esa clase y el **recorrido de su centro** a medida que se mueve. Al final cuenta cuántos IDs distintos aparecieron por clase.

In [ ]:
from collections import defaultdict

import cv2
import numpy as np
from ultralytics.utils.plotting import colors


def guardar_video_tracking(resultados, salida="tracking.mp4", fps=30, largo_cola=40):
    recorridos = defaultdict(list)  # ID -> lista de centros (x, y)
    ids_por_clase = defaultdict(set)  # clase -> IDs vistos
    writer = None

    for r in resultados:
        frame = r.orig_img.copy()
        if writer is None:
            alto, ancho = frame.shape[:2]
            writer = cv2.VideoWriter(salida, cv2.VideoWriter_fourcc(*"mp4v"), fps, (ancho, alto))

        if r.boxes.id is not None:  # None mientras el tracker no confirmó ningún objeto
            for caja, track_id, clase, prob in zip(
                r.boxes.xyxy.int().tolist(), r.boxes.id.int().tolist(),
                r.boxes.cls.int().tolist(), r.boxes.conf.tolist(),
            ):
                x1, y1, x2, y2 = caja
                nombre = r.names[clase]
                color = colors(track_id, bgr=True)
                ids_por_clase[nombre].add(track_id)

                recorridos[track_id].append(((x1 + x2) // 2, (y1 + y2) // 2))
                puntos = np.array(recorridos[track_id][-largo_cola:], dtype=np.int32)

                cv2.rectangle(frame, (x1, y1), (x2, y2), color, 3)
                cv2.polylines(frame, [puntos], isClosed=False, color=color, thickness=4)
                cv2.circle(frame, tuple(puntos[-1]), 8, color, -1)

                texto = f"#{track_id} {nombre} {prob:.2f}"
                (tw, th), _ = cv2.getTextSize(texto, cv2.FONT_HERSHEY_SIMPLEX, 1, 2)
                cv2.rectangle(frame, (x1, y1 - th - 14), (x1 + tw + 10, y1), color, -1)
                cv2.putText(frame, texto, (x1 + 5, y1 - 8), cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 255, 255), 2)

        writer.write(frame)

    if writer is None:
        raise RuntimeError("No llegó ningún frame. Vuelvan a correr la celda 4a: "
                           "con stream=True, resultados se puede recorrer una sola vez.")
    writer.release()
    return ids_por_clase

**c)** Generamos el video y lo convertimos a H.264 para poder verlo en el notebook.

In [ ]:
fps = cv2.VideoCapture(VIDEO).get(cv2.CAP_PROP_FPS)
ids_por_clase = guardar_video_tracking(resultados, "tracking.mp4", fps=fps)

print("IDs distintos por clase:")
for nombre, ids in sorted(ids_por_clase.items()):
    print(f"  {nombre:<20} {len(ids)}  -> IDs {sorted(ids)}")

!ffmpeg -y -loglevel error -i tracking.mp4 -vcodec libx264 -pix_fmt yuv420p tracking_web.mp4
Video("tracking_web.mp4", embed=True, width=800)

**Para pensar**

1. ¿Cada objeto mantiene un único ID durante todo el video? Si algún objeto recibió más de uno, ¿en qué momento cambió?
2. En una caja automática, ¿qué pasaría con el cobro si un producto cambia de ID? Prueben con `"bytetrack.yaml"` y comparen.

**Sus respuestas:**

1. …
2. …